# Notebook level metrics from `jdaviz_profiler`

Read the metrics table:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table

nb_path = 'n_scas/'
notebook_metrics = Table.read('n_scas/medium_notebook_metrics.csv', delimiter=',', format='csv')

Gather column names of metrics and parameters:

In [ ]:
metrics = [name for name in notebook_metrics.colnames if name.endswith('metric')]
params = [name for name in notebook_metrics.colnames if name.endswith('param')]

metrics

Exclude runs that did not complete:

In [ ]:
completed_runs = notebook_metrics['total_cells'] == notebook_metrics['executed_cells']

Make pairwise plots for these parameters, over each of the metrics:

plot `metric` (color) for each combination of `pair_params`:

In [ ]:
pair_params = ['gwcs_to_fits_sip_param', 'n_images_param']
metric = 'kernel_execution_time_metric'

fig, ax = plt.subplots(
    1, 1, figsize=(8, 4),
    sharex='col', sharey='row', layout='constrained'
)
ax = [ax]

vmin, vmax = np.percentile(np.array(notebook_metrics[metric]), [5, 95])
cbar_limits = dict(
    vmin=vmin,
    vmax=vmax
)
param_i, param_j = pair_params
x = np.sort(list(set(notebook_metrics[completed_runs][param_i])))
y = np.sort(list(set(notebook_metrics[completed_runs][param_j])))

grid = np.zeros((len(x), len(y)))
for k in range(len(x)):
    for ell in range(len(y)):
        condition = (
            (x[k] == notebook_metrics[param_i]) & 
            (y[ell] == notebook_metrics[param_j])
        )
        grid[k, ell] = np.median(notebook_metrics[metric][condition])


cax = ax[0].pcolormesh(
    y, x, grid, shading='nearest',
    alpha=1,
    **cbar_limits
)
if not isinstance(x[0], str):
    ax[0].set(
        # ylim=[x.min(), x.max()],
    )

if not isinstance(y[0], str):
    ax[0].set(
        # xlim=[y.min(), y.max()]
    )


ax[0].set(
    ylabel=param_i,
)
        
fig.colorbar(cax, ax=ax[-1], label=metric)
plt.show()

Show how execution time scales with the number of SCAs:

In [ ]:
for metric in ['client', 'kernel']:
    metric_key = f'{metric}_execution_time_metric'
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    for i, server_size in enumerate(['small', 'medium']):
        #server_size = 'medium'
        cell_metrics = Table.read(f'n_scas/{server_size}_cell_metrics.csv', delimiter=',', format='csv')
        load_images = cell_metrics[cell_metrics['cell_index'] == 8]
        color = f'C{i}'
        marker = 'o' if server_size == 'small' else 's'
    
        
        for group in load_images.group_by('gwcs_to_fits_sip_param').groups:
            mask = group['execution_status'] == 'Completed'
            if group['gwcs_to_fits_sip_param'][0] == 'False':
                label = f'GWCS ({server_size})'
            else:
                label = f"GWCS -> FITS SIP ({server_size})"
        
            ax[0].scatter(group['n_images_param'][mask], 
                          group[metric_key][mask], 
                          label=label,
                          marker=marker)
        
        # ax[0].legend(loc='upper left')
        ax[0].set(
            xlabel='# detectors loaded',
            ylabel='time [s]\nto load images',
            ylim=[2, 17],
        )
        
        
        wcs_link_cell = cell_metrics[cell_metrics['cell_index'] == 10]
        for group in wcs_link_cell.group_by('gwcs_to_fits_sip_param').groups:
            mask = group['execution_status'] == 'Completed'
        
            if group['gwcs_to_fits_sip_param'][0] == 'False':
                label = f'GWCS ({server_size})'
            else:
                label = f"GWCS -> FITS SIP ({server_size})"
        
            ax[1].plot(group['n_images_param'][mask], 
                       group[metric_key][mask], 
                       label=label, 
                       marker=marker, ls='-', lw=0.5)
        
        ax[0].legend(loc='upper left')
        ax[1].set(
            xlabel='# detectors loaded',
            ylabel='time [s]\nfor WCS linking',
            # ylim=[2, 17],
            
        )
        ax[1].yaxis.set_label_position("right")
        ax[1].yaxis.tick_right()
        
        ax[1].axvspan(10, 18, alpha=0.2, color='silver')
        
        for axis in ax:
            axis.grid(ls=':')

    fig.suptitle(metric_key)
    fig.tight_layout()
    fig.savefig(f'plots/n_scas_gwcs_{metric}.png', bbox_inches='tight', dpi=250)